In [ ]:
# RQ1: do the published overhead ratios hold on common hardware? (spec 2, H1)
import json, sys
from pathlib import Path

import pandas as pd

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.analysis import frame, plots
from cs4406_sem_project.harness.grid import fmt_size

RUNS = [repo / "results/raw/all-20261112_tier1", repo / "results/raw/all-20261112_anchors"]  # notebook 17
platform = json.loads((RUNS[0] / "meta.json").read_text())["platform"]
df, status = frame.load_runs(RUNS, platform)
t = plots.build(df, status)
FIG = repo / "docs/figures"
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 400)


def save(name, fig):
    for ext in ("pdf", "png"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=200, bbox_inches="tight")


def save_table(name, table):
    (FIG / f"{name}.md").write_text(plots._markdown(table, plots.TABLE_TITLES[name], False), encoding="utf-8")


plots._style()
print(f"{len(df)} measurement rows from {', '.join(r.name for r in RUNS)}; {len(status)} (cell, phase) rows not ok")

In [ ]:
# T3: every target in configs/reproduction_targets.yaml against the grid and the anchors, with S4's extrapolated
# sizes for R8 (decision 10). H1: communication within +-5 % of the published value (rule exact; plot readings
# within a factor 1.2), compute within the published range up to a hardware factor (rule hw).
t3 = t.t3
rows = t3[t3.verdict != "see summary row"]
print(rows.verdict.value_counts().to_string(), "\n")
for rule in ("exact", "hw"):
    r = rows[rows.rule == rule]
    print(f"rule {rule}: {len(r)} rows, verdicts {r.verdict.value_counts().to_dict()}")
print()
print(t3[["id", "cell", "ours", "ci_lo", "ci_hi", "unit", "target", "deviation", "verdict"]].round(3).to_string(index=False))
save_table("T3_reproduction", t3.drop(columns=["check", "target_lo", "target_hi"]))

In [ ]:
# The rows H1 does not pass, with what each needs: "explain" is outside tolerance on our hardware; "missing" has
# no row (a cell that ran out of memory, or a size we do not run).
odd = t3[t3.verdict.isin(["explain", "missing"])]
print(odd[["id", "claim", "cell", "ours", "target", "deviation", "verdict"]].round(3).to_string(index=False))
print("\ncells not ok in the runs:")
print(status.groupby(["arm", "status"]).size().to_string())

In [ ]:
# F1: published against measured, per claim, with 95 % CIs.
fig = plots.f1_reproduction(t)
save("F1_reproduction", fig)
print(sorted(p.name for p in FIG.glob("F1_*")))